In [30]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

In [31]:
class MyState(TypedDict):
    recipient: str
    amount: int
    memo: str
    approved: bool
    final_status: str

# 审查节点
def review_node(state: MyState) -> MyState:
    # 正在执行的转账信息
    pending_transfer = {
        "recipient": state["recipient"],
        "amount": state["amount"],
        "memo": state["memo"]
    }

    res = interrupt(
        {
            "title": "转账请求",
            "pending_transfer": pending_transfer,
            "instruction": "请返回True或False或修改的内容(recipient, amount, memo, approved)"
        }
    )    

    new_info = {}

    if isinstance(res, bool):
        return {"approved": res}
    else:
        return res

def output_node(state: MyState) -> MyState:
    if state["approved"]:
        return {
            "final_status": (
                f"转账成功，转账信息："
                f"收款人: {state['recipient']}, "
                f"金额: {state['amount']}, "
                f"备注: {state['memo']}"
            )
        }

    return {
        "final_status": "转账请求被拒绝"
    }

builder = StateGraph(MyState)

builder.add_node(review_node)
builder.add_node(output_node)

builder.add_edge(START, "review_node")
builder.add_edge("review_node", "output_node")
builder.add_edge("output_node", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {
    "configurable": {
        "thread_id": "123"
    }
}

inputs = {
    "recipient": "张三",
    "amount": 100,
    "memo": "测试1"
}

interrupt_res = graph.invoke(input=inputs, config=config)

print(interrupt_res)

{'recipient': '张三', 'amount': 100, 'memo': '测试1', '__interrupt__': [Interrupt(value={'title': '转账请求', 'pending_transfer': {'recipient': '张三', 'amount': 100, 'memo': '测试1'}, 'instruction': '请返回True或False或修改的内容(recipient, amount, memo, approved)'}, id='36404880f43ba5be5a308467d0a10d91')]}


In [32]:
if interrupt_res.get("__interrupt__"):
    interrupt_infos = interrupt_res["__interrupt__"]

    for interrupt_info in interrupt_infos:
        print(interrupt_info.value)

{'title': '转账请求', 'pending_transfer': {'recipient': '张三', 'amount': 100, 'memo': '测试1'}, 'instruction': '请返回True或False或修改的内容(recipient, amount, memo, approved)'}


In [33]:
review_info = {
    "approved": True,
    "amount": int(input("请输入转账金额"))
}
res = graph.invoke(Command(resume=review_info), config=config)
print(res)

{'recipient': '张三', 'amount': 300, 'memo': '测试1', 'approved': True, 'final_status': '转账成功，转账信息：收款人: 张三, 金额: 300, 备注: 测试1'}
